## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.cryptography.optimized_p_cnn_ipfe import IPFEPaillier
from src.utils.notebook_helper import encrypt_test_data, test_ipfe_cnn, test_regular_ipfe_cnn, load_data

In [3]:
base_model = 1
model_path = f"models/cnn_model_{base_model}.pth"

## Define Model

In [4]:
class IPFECNN(nn.Module):
    def __init__(self, device, num_classes=10):
        super(IPFECNN, self).__init__()
        self.ipfe = IPFEPaillier(n_length=48, max_workers=None)
        self.encryption_length = 9 # 3x3 filter size flattened
        self.device = device

        self.ipfe.setup(l=self.encryption_length)
        print("IPFE setup done, with length:", self.encryption_length)

        # First convolutional block - this will be used with IPFE
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=3, padding=1) # stride = 2, padding = 0
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Second convolutional block
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Third convolutional block
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 1 * 1, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

        #copy weights from the trained model
        self.load_state_dict(torch.load(model_path, map_location=device))
        print("weights copied from trained model")

        self.weights = self.conv1.weight.data
        self.y_array = torch.round(self.weights.view(self.weights.size(0), -1).squeeze(1).view(self.weights.size(0), -1) * 10000).long().tolist()
        print("weights converted to y vectors")
        self.biases = self.conv1.bias
        print("biases saved")
        self.sk_y_array = self.ipfe.key_derive_batch(self.y_array)
        print("sk_ys created")

    def encrypt_data(self, test_set):
        unfold = nn.Unfold(kernel_size=3, stride=3, padding=1)
        patches = unfold(test_set)  # (B, 9, num_patches)
        B, patch_size, num_patches = patches.shape

        encrypted_patches = []
        for b in range(B):
            patches_b = patches[b].T  # (num_patches, 9)
            encrypted_image = []
            for p in range(num_patches):
                patch = patches_b[p]  # (9,)
                ct_patch = self.ipfe.encrypt(patch)
                encrypted_image.append(ct_patch)
            encrypted_patches.append(encrypted_image)

        return encrypted_patches

    def first_conv_forward(self, encrypted_image, H, W):
        num_patches = len(encrypted_image)
        num_kernels = len(self.sk_y_array)

        # Use optimized batch decrypt: 16 threads (one per kernel) instead of 500+ tasks
        print(len(encrypted_image))
        print(len(encrypted_image[0]))
        print(len(encrypted_image[0][1]))
        decrypted_vals = self.ipfe.decrypt_patches_kernels_batch_optimized(encrypted_image, self.sk_y_array, self.y_array, scale=10000)

        # decrypted_vals is (num_patches, num_kernels), transpose to (num_kernels, num_patches)
        decrypted_maps = torch.tensor(decrypted_vals.T, dtype=torch.float32, device=self.device) + self.biases.unsqueeze(1)

        # Reshape to (1, num_kernels, H, W)
        return decrypted_maps.view(1, num_kernels, H, W)

    def forward(self, x, H, W, encrypted=False):
        if encrypted:
            outputs = []
            for sample in x:  # x = [ [patches_img1], [patches_img2], ... ]
                feat = self.first_conv_forward(sample, H, W)
                feat = self.pool1(F.relu(self.bn1(feat)))
                feat = self.pool2(F.relu(self.bn2(self.conv2(feat))))
                feat = self.pool3(F.relu(self.bn3(self.conv3(feat))))
                feat = feat.view(feat.size(0), -1)
                feat = F.relu(self.fc1(feat))
                feat = self.dropout(feat)
                feat = self.fc2(feat)
                outputs.append(feat)
            return torch.cat(outputs, dim=0)
        else:
            x = self.conv1(x)
            x = self.pool1(F.relu(self.bn1(x)))
            x = self.pool2(F.relu(self.bn2(self.conv2(x))))
            x = self.pool3(F.relu(self.bn3(self.conv3(x))))
            x = x.view(x.size(0), -1)
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            return x



## Initialize Model

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ipfe_model = IPFECNN(device=device, num_classes=10).to(device)

print(f"IPFE-CNN model created on device: {device}")

=== IPFEPaillier Setup ===
  n_length  : 48 bits
  N         : 48 bits
  l         : 9
  X_bound   : 255
  Y_bound   : 10000
  max|<x,y>|: 22950000  <  N ok
IPFE setup done, with length: 9
weights copied from trained model
weights converted to y vectors
biases saved
sk_ys created
IPFE-CNN model created on device: cpu


In [6]:
test_loader = load_data()
encrypted_data, labels = encrypt_test_data(ipfe_model, test_loader, device, num_samples=1)

Test samples: 10000
Encrypted 1 samples.


## Test model

In [7]:
print("Testing IPFE-CNN functionality...")
test_ipfe_cnn(ipfe_model, encrypted_data, labels, H=10, W=10, device=device)


Testing IPFE-CNN functionality...
Testing CNN forward pass on encrypted data...
Labels of test samples: [7]
100
2
9
number of patches:  100
example ct:  (32015505822297544899968426739, [868450515718486680704228129, 31856680204465463326304150636, 9997053579411760293563738411, 4789837370121603361244814648, 27836183322376181303476276417, 19284898848569554181544923774, 14283641647873382313321807574, 5652923998963847324667473827, 28777202033470035806666014301])
Predictions on encrypted data: [7]
Accuracy on encrypted samples: 100.00% (1/1)


In [7]:
test_regular_ipfe_cnn(ipfe_model, test_loader, device, num_samples=5)

Testing CNN forward pass on encrypted data...
Predictions on encrypted data: [7 2 1 0 4]
Accuracy on encrypted samples: 100.00% (5/5)
